# LocalRetro 模型展示教程

## 概述

本 notebook 深入剖析 **LocalRetro** 的模型计算架构和推理原理。

LocalRetro 的核心思想是将逆合成预测转化为**原子级别和键级别的分类任务**：

$$P(\text{reactants} | \text{product}) = \text{Decode}\left(\underset{i}{\arg\max}\; P(t_i^{\text{atom}} | p),\; \underset{j}{\arg\max}\; P(t_j^{\text{bond}} | p)\right)$$

其中：
- $P(t_i^{\text{atom}} | p)$：产物 $p$ 中第 $i$ 个原子的局部模板概率
- $P(t_j^{\text{bond}} | p)$：产物 $p$ 中第 $j$ 条键的局部模板概率
- $\text{Decode}$：将预测的局部模板应用到产物上，恢复反应物

### 与 GLN 的架构对比

| 方面 | GLN | LocalRetro |
|------|-----|------------|
| 预测目标 | 层次化: 中心→模板→验证 | **直接**: 每个原子/键 → 模板类别 |
| GNN | 自定义 Mean-Field GNN | **MPNN** (DGLLife) |
| 注意力机制 | 产物-候选项 注意力 | **Global Reactivity Attention** |
| 损失函数 | 三级 NLL Loss | **CrossEntropyLoss** |
| 推理方式 | Beam Search + 模板应用 | Top-K 编辑 + 模板解码 |

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 分子图的特征编码（DGLLife Featurizers） |
| 2 | MPNN 消息传递编码器 |
| 3 | 键特征构建（pair_atom_feats） |
| 4 | 编辑特征组装与掩码（unbatch_mask / unbatch_feats） |
| 5 | Global Reactivity Attention |
| 6 | 原子/键分类头 |
| 7 | 完整的 LocalRetro 模型 |
| 8 | 推理流程端到端演示 |
| 9 | 训练流程概览 |

> **说明**: 本教程用最小化的纯 Python/PyTorch/DGL 代码重现 LocalRetro 的核心计算逻辑，**不依赖 LocalRetro 原始代码的 import**。所有模块均面向教学设计，仅用于教学演示。

---

In [1]:
# ====== 基础导入 ======
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from functools import partial

import dgl
from dgllife.model import MPNNGNN
from dgllife.utils import (
    WeaveAtomFeaturizer, 
    CanonicalBondFeaturizer, 
    smiles_to_bigraph
)
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdChemReactions
from IPython.display import HTML, display

DEVICE = torch.device("cpu")
print(f'计算设备: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'DGL: {dgl.__version__}')
print(f'所有导入成功！')

计算设备: cpu
PyTorch: 2.4.0+cu121
DGL: 2.4.0
所有导入成功！


---

## 1. 分子图的特征编码

LocalRetro 使用 DGLLife 内置的 **WeaveAtomFeaturizer** 和 **CanonicalBondFeaturizer** 来提取原子/键特征。

与 GLN 手工定义位打包特征 + C++ 展开不同，LocalRetro 直接使用高层 API 生成 one-hot 向量。

### WeaveAtomFeaturizer 特征组成

```
原子特征向量 (27维):
┌─────────────┬──────┬──────┬──────┬──────┬───────┐
│ 原子类型(65) │ 手性(5) │ 度数(7) │ 电荷(6) │ 氢数(6) │ 芳香(1) │
└─────────────┴──────┴──────┴──────┴──────┴───────┘
  → 通过 WeaveAtomFeaturizer 自动生成
```

### CanonicalBondFeaturizer 特征组成

```
键特征向量 (12维):
┌──────────┬──────┬──────┬──────────┐
│ 键类型(4) │ 共轭(1) │ 环(1) │ 立体(6)  │
└──────────┴──────┴──────┴──────────┘
  → 通过 CanonicalBondFeaturizer 自动生成
  → self_loop=True: 自环的特征全为零
```

### LocalRetro 源码对应
- `scripts/utils.py` → `init_featurizer()`

In [2]:
# ====== 1. 特征编码器 ======
# 对应 LocalRetro 源码: scripts/utils.py → init_featurizer()

atom_types = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
         'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
         'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
         'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
         'Ce', 'Gd', 'Ga', 'Cs']

node_featurizer = WeaveAtomFeaturizer(atom_types=atom_types)
edge_featurizer = CanonicalBondFeaturizer(self_loop=True)

NODE_FEAT_DIM = node_featurizer.feat_size()
EDGE_FEAT_DIM = edge_featurizer.feat_size()

print(f'节点特征维度: {NODE_FEAT_DIM}')
print(f'边特征维度:   {EDGE_FEAT_DIM}')

# 构建示例分子图
smiles_to_graph = partial(smiles_to_bigraph, add_self_loop=True)
demo_mol = 'CC(=O)Nc1ccccc1'  # 乙酰苯胺
g = smiles_to_graph(demo_mol, node_featurizer=node_featurizer,
                    edge_featurizer=edge_featurizer, canonical_atom_order=False)

mol = Chem.MolFromSmiles(demo_mol)
print(f'\n分子: {demo_mol}')
print(f'原子数: {g.num_nodes()}, 边数: {g.num_edges()} (含 {g.num_nodes()} 条自环)')
print(f'节点特征: {g.ndata["h"].shape}')
print(f'边特征:   {g.edata["e"].shape}')

# 查看各原子特征
print(f'\n各原子特征摘要:')
for i, atom in enumerate(mol.GetAtoms()):
    feat = g.ndata['h'][i]
    nonzero = torch.nonzero(feat).flatten().tolist()
    print(f'  原子 {i} ({atom.GetSymbol():2s}): 非零特征索引 = {nonzero}')

节点特征维度: 80
边特征维度:   13

分子: CC(=O)Nc1ccccc1
原子数: 10, 边数: 30 (含 10 条自环)
节点特征: torch.Size([10, 80])
边特征:   torch.Size([30, 13])

各原子特征摘要:
  原子 0 (C ): 非零特征索引 = [0, 67, 71]
  原子 1 (C ): 非零特征索引 = [0, 67, 70]
  原子 2 (O ): 非零特征索引 = [2, 67, 70, 73]
  原子 3 (N ): 非零特征索引 = [1, 67, 70, 72]
  原子 4 (C ): 非零特征索引 = [0, 67, 68, 70, 77]
  原子 5 (C ): 非零特征索引 = [0, 67, 68, 70, 77]
  原子 6 (C ): 非零特征索引 = [0, 67, 68, 70, 77]
  原子 7 (C ): 非零特征索引 = [0, 67, 68, 70, 77]
  原子 8 (C ): 非零特征索引 = [0, 67, 68, 70, 77]
  原子 9 (C ): 非零特征索引 = [0, 67, 68, 70, 77]


---

## 2. MPNN 消息传递编码器

LocalRetro 使用 DGLLife 内置的 **MPNNGNN**（Message Passing Neural Network）作为分子编码器。

### MPNN 的计算流程

MPNN 在**边上**进行消息传递（与 GLN 的 Mean-Field GNN 不同，后者在节点上传递）：

$$\begin{aligned}
m_{uv}^{(t)} &= \text{EdgeMLP}\left(h_u^{(t)}, e_{uv}\right) \\
h_v^{(t+1)} &= \text{GRU}\left(h_v^{(t)}, \sum_{u \in \mathcal{N}(v)} m_{uv}^{(t)}\right)
\end{aligned}$$

其中：
- $h_v^{(t)}$：第 $t$ 步节点 $v$ 的嵌入
- $e_{uv}$：边 $(u, v)$ 的特征
- $\text{EdgeMLP}$：边消息函数（MLP）
- $\text{GRU}$：门控循环单元（更新节点状态）

### 与 GLN Mean-Field GNN 的区别

| 方面 | GLN Mean-Field | LocalRetro MPNN |
|------|----------------|-----------------|
| 消息计算 | 聚合邻居节点特征 | **使用边特征**参与消息计算 |
| 节点更新 | 线性变换 + tanh | **GRU** 门控更新 |
| 实现 | 自定义 MessagePassing | DGLLife **内置** MPNNGNN |

### LocalRetro 源码对应
- `scripts/models.py` → `self.mpnn = MPNNGNN(...)`

In [3]:
# ====== 2. MPNN 编码器 ======
# 对应 LocalRetro 源码: scripts/models.py → self.mpnn

# 配置参数（对应 default_config.json）
NODE_OUT_FEATS = 320       # 节点输出特征维度
EDGE_HIDDEN_FEATS = 64     # 边隐藏层维度
NUM_STEP_MESSAGE_PASSING = 6  # 消息传递步数

# 创建 MPNN
mpnn = MPNNGNN(
    node_in_feats=NODE_FEAT_DIM,
    node_out_feats=NODE_OUT_FEATS,
    edge_in_feats=EDGE_FEAT_DIM,
    edge_hidden_feats=EDGE_HIDDEN_FEATS,
    num_step_message_passing=NUM_STEP_MESSAGE_PASSING
)

print('MPNN 架构:')
print(f'  输入: 节点({NODE_FEAT_DIM}维) + 边({EDGE_FEAT_DIM}维)')
print(f'  输出: 节点({NODE_OUT_FEATS}维)')
print(f'  消息传递步数: {NUM_STEP_MESSAGE_PASSING}')
print(f'  参数量: {sum(p.numel() for p in mpnn.parameters()):,}')

# 前向传播演示
node_feats = g.ndata['h']
edge_feats = g.edata['e']
with torch.no_grad():
    node_out = mpnn(g, node_feats, edge_feats)

print(f'\n前向传播:')
print(f'  输入节点特征: {node_feats.shape}')
print(f'  输出节点特征: {node_out.shape}')
print(f'  每个原子现在有 {NODE_OUT_FEATS} 维的上下文感知嵌入')

MPNN 架构:
  输入: 节点(80维) + 边(13维)
  输出: 节点(320维)
  消息传递步数: 6
  参数量: 7,299,456

前向传播:
  输入节点特征: torch.Size([10, 80])
  输出节点特征: torch.Size([10, 320])
  每个原子现在有 320 维的上下文感知嵌入


---

## 3. 键特征构建 (pair_atom_feats)

### 核心思想

MPNN 编码后，我们得到了每个**原子**的嵌入向量。但 LocalRetro 还需要预测**键**的编辑操作，因此需要为每条键构建特征。

方法很简单：将键两端原子的嵌入**拼接 (concatenate)**，然后通过线性层降维：

$$\mathbf{e}_{uv} = \text{Linear}\left([\mathbf{h}_u \| \mathbf{h}_v]\right)$$

### 注意事项
- 使用 `remove_self_loop()` 去除自环，因为自环不是真正的化学键
- 键是有向边 (u→v)，所以每条化学键会生成两个方向的特征

### LocalRetro 源码对应
- `scripts/model_utils.py` → `pair_atom_feats()`
- `scripts/models.py` → `self.linearB`

In [4]:
# ====== 3. 键特征构建 ======
# 对应 LocalRetro 源码: scripts/model_utils.py → pair_atom_feats()

def pair_atom_feats(g, node_feats):
    """
    将键两端原子的嵌入拼接，构建键级别特征。
    
    对应 LocalRetro 源码: model_utils.py → pair_atom_feats()
    
    输入: g (DGL图，含自环), node_feats [n_atoms, dim]
    输出: bond_feats [n_edges_no_selfloop, dim*2]
    """
    sg = g.remove_self_loop()  # 去除自环
    atom_idx1, atom_idx2 = sg.edges()
    # 拼接键两端原子的特征
    bond_feats = torch.cat([
        node_feats[atom_idx1.long()], 
        node_feats[atom_idx2.long()]
    ], dim=1)
    return bond_feats

# 演示
with torch.no_grad():
    atom_feats = mpnn(g, g.ndata['h'], g.edata['e'])  # MPNN 输出
    bond_feats_raw = pair_atom_feats(g, atom_feats)    # 拼接

print(f'MPNN 输出的原子特征: {atom_feats.shape}')
print(f'去自环后的边数: {g.remove_self_loop().num_edges()}')
print(f'拼接后的键特征: {bond_feats_raw.shape} (dim = {NODE_OUT_FEATS} × 2)')

# 线性降维
linearB = nn.Linear(NODE_OUT_FEATS * 2, NODE_OUT_FEATS)
with torch.no_grad():
    bond_feats = linearB(bond_feats_raw)

print(f'降维后的键特征: {bond_feats.shape} (dim = {NODE_OUT_FEATS})')
print()
print('此时我们有:')
print(f'  原子特征: {atom_feats.shape[0]} 个原子 × {atom_feats.shape[1]} 维')
print(f'  键特征:   {bond_feats.shape[0]} 条键 × {bond_feats.shape[1]} 维')

MPNN 输出的原子特征: torch.Size([10, 320])
去自环后的边数: 20
拼接后的键特征: torch.Size([20, 640]) (dim = 320 × 2)
降维后的键特征: torch.Size([20, 320]) (dim = 320)

此时我们有:
  原子特征: 10 个原子 × 320 维
  键特征:   20 条键 × 320 维


---

## 4. 编辑特征组装与掩码

### 核心思想

LocalRetro 的关键设计之一是将**原子特征和键特征**拼接成一个统一的**编辑特征序列 (edit features)**，然后用注意力机制让所有编辑位点互相交互。

对于每个分子：
```
编辑特征序列 = [原子0, 原子1, ..., 原子n, 键0→1, 键1→0, 键0→2, ...]
                ├── 原子特征 ──┤├────── 键特征 ──────┤
```

由于不同分子的原子数和键数不同，需要用 **padding + mask** 来处理变长序列（类似 NLP 中处理变长文本）。

### 处理流程

```
batch 图 (bg)
  ↓ unbatch → 拆分为单个图
  ↓ 拼接原子特征 + 键特征 → 编辑特征
  ↓ pad_sequence → 等长序列
  ↓ 生成 mask (1=有效, 0=padding)
  ↓ Global Reactivity Attention
  ↓ unbatch_feats → 拆分回原子特征和键特征
```

### LocalRetro 源码对应
- `scripts/model_utils.py` → `unbatch_mask()`, `unbatch_feats()`

In [5]:
# ====== 4. 编辑特征组装 ======
# 对应 LocalRetro 源码: scripts/model_utils.py → unbatch_mask(), unbatch_feats()

def unbatch_mask(bg, atom_feats, bond_feats):
    """
    将 batch 图的原子/键特征拆分并组装为编辑特征序列。
    
    对应 LocalRetro 源码: model_utils.py → unbatch_mask()
    
    步骤:
    1. 去除自环，拆分 batch 图为单个图
    2. 对每个图：拼接 [原子特征; 键特征] → 编辑特征
    3. 对编辑特征做 padding（补零到最长序列）
    4. 生成 mask（标记有效位置）
    
    返回:
        edit_feats: [batch_size, max_seq_len, dim]
        masks: [batch_size, max_seq_len] (1=有效, 0=padding)
    """
    edit_feats = []
    masks = []
    sg = bg.remove_self_loop()
    sg.ndata['h'] = atom_feats
    sg.edata['e'] = bond_feats
    gs = dgl.unbatch(sg)
    
    for g_single in gs:
        # 拼接原子特征和键特征
        e_feats = torch.cat([g_single.ndata['h'], g_single.edata['e']], dim=0)
        mask = torch.ones(e_feats.size(0), dtype=torch.uint8)
        edit_feats.append(e_feats)
        masks.append(mask)
    
    # Padding 到相同长度
    edit_feats = pad_sequence(edit_feats, batch_first=True, padding_value=0)
    masks = pad_sequence(masks, batch_first=True, padding_value=0)
    
    return edit_feats, masks

def unbatch_feats(bg, edit_feats):
    """
    将注意力处理后的编辑特征拆分回原子特征和键特征。
    
    对应 LocalRetro 源码: model_utils.py → unbatch_feats()
    """
    sg = bg.remove_self_loop()
    gs = dgl.unbatch(sg)
    atom_feats_list = []
    bond_feats_list = []
    
    for i, g_single in enumerate(gs):
        n_atoms = g_single.num_nodes()
        n_bonds = g_single.num_edges()
        atom_feats_list.append(edit_feats[i][:n_atoms])
        bond_feats_list.append(edit_feats[i][n_atoms:n_atoms + n_bonds])
    
    return torch.cat(atom_feats_list, dim=0), torch.cat(bond_feats_list, dim=0)

# 演示
smiles_list = ['CC(=O)Nc1ccccc1', 'c1ccccc1']
graphs = [smiles_to_graph(s, node_featurizer=node_featurizer, 
                          edge_featurizer=edge_featurizer,
                          canonical_atom_order=False) for s in smiles_list]
bg = dgl.batch(graphs)

with torch.no_grad():
    all_node_feats = mpnn(bg, bg.ndata['h'], bg.edata['e'])
    all_bond_feats = linearB(pair_atom_feats(bg, all_node_feats))
    edit_feats, masks = unbatch_mask(bg, all_node_feats, all_bond_feats)

print(f'Batch 统计:')
sg = bg.remove_self_loop()
for i, g_single in enumerate(dgl.unbatch(sg)):
    seq_len = g_single.num_nodes() + g_single.num_edges()
    print(f'  分子 {i} ({smiles_list[i]}): {g_single.num_nodes()} 原子 + {g_single.num_edges()} 键 = {seq_len} 编辑位点')

print(f'\nPadding 后的编辑特征: {edit_feats.shape}')
print(f'Mask: {masks.shape}')
print(f'  分子 0 mask: {masks[0].tolist()[:20]}... (1=有效)')
print(f'  分子 1 mask: {masks[1].tolist()[:20]}... (0=padding)')

Batch 统计:
  分子 0 (CC(=O)Nc1ccccc1): 10 原子 + 20 键 = 30 编辑位点
  分子 1 (c1ccccc1): 6 原子 + 12 键 = 18 编辑位点

Padding 后的编辑特征: torch.Size([2, 30, 320])
Mask: torch.Size([2, 30])
  分子 0 mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]... (1=有效)
  分子 1 mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]... (0=padding)


---

## 5. Global Reactivity Attention

### 核心创新

Global Reactivity Attention (GRA) 是 LocalRetro 的**核心创新点**。它让所有编辑位点（原子 + 键）能够互相关注，从而捕捉**全局反应性模式**。

### 为什么需要全局注意力？

MPNN 只能聚合**局部邻居**的信息（受消息传递步数限制）。但化学反应中，一个位点的反应性可能受远处基团的影响（如电子效应）。GRA 通过全局注意力让每个编辑位点都能"看到"分子中的所有其他位点。

### 架构

GRA 使用标准的 **Multi-Head Self-Attention + Feed-Forward** 结构（类似 Transformer Encoder）：

$$\begin{aligned}
\text{MultiHead}(Q, K, V) &= \text{Concat}(\text{head}_1, ..., \text{head}_h) \\
\text{head}_i &= \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V) \\
\text{Attention}(Q, K, V) &= \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V \\
\text{FFN}(x) &= \text{Linear}(\text{GELU}(\text{Linear}(x)))
\end{aligned}$$

### 配置参数
- `attention_heads = 8`: 注意力头数
- `attention_layers = 1`: 注意力层数（默认只用 1 层）

### LocalRetro 源码对应
- `scripts/model_utils.py` → `MultiHeadAttention`, `FeedForward`, `Global_Reactivity_Attention`

In [6]:
# ====== 5.1 GELU 激活函数 ======
# 对应 LocalRetro 源码: model_utils.py → GELU

class GELU(nn.Module):
    """
    GELU (Gaussian Error Linear Unit) 激活函数。
    LocalRetro 自定义实现（兼容旧版 PyTorch）。
    """
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))

# ====== 5.2 多头自注意力 ======
# 对应 LocalRetro 源码: model_utils.py → MultiHeadAttention

class MultiHeadAttention(nn.Module):
    """
    多头自注意力机制。
    
    对应 LocalRetro 源码: model_utils.py → MultiHeadAttention
    
    与标准 Transformer 注意力的区别:
    - 使用 mask 处理变长序列（不同分子的编辑位点数不同）
    - 返回注意力分数（用于可视化）
    - 使用残差连接 + LayerNorm
    """
    def __init__(self, heads, d_model, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.d_k = d_model // heads
        self.h = heads
        self.q_linear = nn.Linear(d_model, d_model, bias=False)
        self.v_linear = nn.Linear(d_model, d_model, bias=False)
        self.k_linear = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)
        self._reset_parameters()
    
    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def attention(self, q, k, v, mask=None):
        """
        缩放点积注意力。
        
        scores = softmax(Q·K^T / √d_k) · V
        
        mask 处理:
        - 将 padding 位置的分数设为 -9e15（近似 -∞）
        - softmax 后这些位置的权重接近 0
        """
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # 扩展 mask 到注意力矩阵的形状 [batch, heads, seq, seq]
            mask_expanded = mask.unsqueeze(1).repeat(1, mask.size(-1), 1)
            mask_expanded = mask_expanded.unsqueeze(1).repeat(1, scores.size(1), 1, 1)
            scores[~mask_expanded.bool()] = float(-9e15)
        scores = torch.softmax(scores, dim=-1)
        scores = self.dropout(scores)
        output = torch.matmul(scores, v)
        return scores, output
    
    def forward(self, x, mask=None):
        bs = x.size(0)
        # 线性变换 + reshape 为多头
        k = self.k_linear(x).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        q = self.q_linear(x).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        v = self.v_linear(x).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        
        scores, output = self.attention(q, k, v, mask)
        
        # 合并多头 + 残差连接 + LayerNorm
        output = output.transpose(1, 2).contiguous().view(bs, -1, self.d_model)
        output = output + x  # 残差连接
        output = self.layer_norm(output)
        return scores, output.squeeze(-1)

print('MultiHeadAttention 创建成功')
print(f'  d_model={NODE_OUT_FEATS}, heads=8, d_k={NODE_OUT_FEATS//8}')

MultiHeadAttention 创建成功
  d_model=320, heads=8, d_k=40


In [7]:
# ====== 5.3 前馈网络 ======
# 对应 LocalRetro 源码: model_utils.py → FeedForward

class FeedForward(nn.Module):
    """
    位置前馈网络 (Position-wise Feed-Forward Network)。
    
    对应 LocalRetro 源码: model_utils.py → FeedForward
    
    结构: Linear(d→2d) → GELU → Linear(2d→d) → Dropout
    + 残差连接 + LayerNorm
    """
    def __init__(self, d_model, activation=GELU(), dropout=0.1):
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            activation,
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)
    
    def forward(self, x):
        output = self.net(x)
        return self.layer_norm(x + output)  # 残差 + LayerNorm

# ====== 5.4 Global Reactivity Attention ======
# 对应 LocalRetro 源码: model_utils.py → Global_Reactivity_Attention

class Global_Reactivity_Attention(nn.Module):
    """
    全局反应性注意力模块。
    
    对应 LocalRetro 源码: model_utils.py → Global_Reactivity_Attention
    
    结构: N 层 × (MultiHeadAttention + FeedForward)
    
    这是 LocalRetro 的核心创新：
    让分子中所有编辑位点（原子+键）互相关注，
    从而捕捉远程电子效应等全局反应性模式。
    """
    def __init__(self, d_model, heads, n_layers=1, dropout=0.1, activation=GELU()):
        super(Global_Reactivity_Attention, self).__init__()
        self.n_layers = n_layers
        att_stack = []
        pff_stack = []
        for _ in range(n_layers):
            att_stack.append(MultiHeadAttention(heads, d_model, dropout))
            pff_stack.append(FeedForward(d_model, activation, dropout))
        self.att_stack = nn.ModuleList(att_stack)
        self.pff_stack = nn.ModuleList(pff_stack)
    
    def forward(self, x, mask):
        scores = []
        for n in range(self.n_layers):
            score, x = self.att_stack[n](x, mask)
            x = self.pff_stack[n](x)
            scores.append(score)
        return scores, x

# 演示
ATTENTION_HEADS = 8
ATTENTION_LAYERS = 1

gra = Global_Reactivity_Attention(
    d_model=NODE_OUT_FEATS,
    heads=ATTENTION_HEADS,
    n_layers=ATTENTION_LAYERS
)

print(f'Global Reactivity Attention:')
print(f'  d_model={NODE_OUT_FEATS}, heads={ATTENTION_HEADS}, layers={ATTENTION_LAYERS}')
print(f'  参数量: {sum(p.numel() for p in gra.parameters()):,}')

# 前向传播
with torch.no_grad():
    attention_scores, attended_feats = gra(edit_feats, masks)

print(f'\n输入编辑特征: {edit_feats.shape}')
print(f'输出编辑特征: {attended_feats.shape}')
print(f'注意力分数: {attention_scores[0].shape}  (batch × heads × seq × seq)')

Global Reactivity Attention:
  d_model=320, heads=8, layers=1
  参数量: 719,040

输入编辑特征: torch.Size([2, 30, 320])
输出编辑特征: torch.Size([2, 30, 320])
注意力分数: torch.Size([2, 8, 30, 30])  (batch × heads × seq × seq)


---

## 6. 原子/键分类头

### 预测逻辑

经过 GRA 增强后的编辑特征被拆分回原子特征和键特征，分别送入两个独立的分类头：

```
注意力增强后的编辑特征
  ↓ unbatch_feats (拆分)
  ├── 原子特征 → atom_linear → [n_atoms, n_atom_templates + 1]
  └── 键特征   → bond_linear → [n_bonds, n_bond_templates + 1]
```

每个分类头的结构：`Linear → GELU → Dropout → Linear`

类别数 = **模板数 + 1**（+1 是「无编辑」类，即 class 0）

### 训练目标

使用标准的 **CrossEntropyLoss**，对每个原子和每条键分别计算损失：

$$\mathcal{L} = \frac{1}{N_a + N_b} \left( \sum_{i} \text{CE}(\hat{y}_i^{\text{atom}}, y_i^{\text{atom}}) + \sum_{j} \text{CE}(\hat{y}_j^{\text{bond}}, y_j^{\text{bond}}) \right)$$

### LocalRetro 源码对应
- `scripts/models.py` → `self.atom_linear`, `self.bond_linear`

In [8]:
# ====== 6. 分类头 ======
# 对应 LocalRetro 源码: scripts/models.py → self.atom_linear, self.bond_linear

# 实际模板数量 (USPTO_50K)
ATOM_TEMPLATE_N = 124  # 原子模板数
BOND_TEMPLATE_N = 548  # 键模板数

activation = GELU()

# 原子分类头: 预测每个原子的 atom template class
atom_linear = nn.Sequential(
    nn.Linear(NODE_OUT_FEATS, NODE_OUT_FEATS),
    activation,
    nn.Dropout(0.2),
    nn.Linear(NODE_OUT_FEATS, ATOM_TEMPLATE_N + 1)  # +1 for "no edit"
)

# 键分类头: 预测每条键的 bond template class
bond_linear = nn.Sequential(
    nn.Linear(NODE_OUT_FEATS, NODE_OUT_FEATS),
    activation,
    nn.Dropout(0.2),
    nn.Linear(NODE_OUT_FEATS, BOND_TEMPLATE_N + 1)  # +1 for "no edit"
)

print(f'原子分类头: 输入 {NODE_OUT_FEATS} → 输出 {ATOM_TEMPLATE_N + 1} (124 模板 + 1 无编辑)')
print(f'键分类头:   输入 {NODE_OUT_FEATS} → 输出 {BOND_TEMPLATE_N + 1} (548 模板 + 1 无编辑)')
print(f'\n原子分类头参数: {sum(p.numel() for p in atom_linear.parameters()):,}')
print(f'键分类头参数:   {sum(p.numel() for p in bond_linear.parameters()):,}')

# 演示
with torch.no_grad():
    atom_feats_out, bond_feats_out = unbatch_feats(bg, attended_feats)
    atom_logits = atom_linear(atom_feats_out)
    bond_logits = bond_linear(bond_feats_out)

print(f'\n前向传播结果:')
print(f'  原子 logits: {atom_logits.shape}  (每个原子预测 {ATOM_TEMPLATE_N+1} 个类)')
print(f'  键 logits:   {bond_logits.shape}  (每条键预测 {BOND_TEMPLATE_N+1} 个类)')

原子分类头: 输入 320 → 输出 125 (124 模板 + 1 无编辑)
键分类头:   输入 320 → 输出 549 (548 模板 + 1 无编辑)

原子分类头参数: 142,845
键分类头参数:   278,949

前向传播结果:
  原子 logits: torch.Size([16, 125])  (每个原子预测 125 个类)
  键 logits:   torch.Size([32, 549])  (每条键预测 549 个类)


---

## 7. 完整的 LocalRetro 模型

将以上所有组件组装为完整的 `LocalRetro_model`：

```
输入: DGL batch 图 (bg) + 节点特征 + 边特征
  │
  ├── (1) MPNN: 消息传递编码 → 原子嵌入 [n_atoms, 320]
  │
  ├── (2) pair_atom_feats: 键两端原子拼接 → 键嵌入 [n_bonds, 640]
  │     linearB: 降维 → [n_bonds, 320]
  │
  ├── (3) unbatch_mask: 组装编辑特征序列 + 掩码
  │
  ├── (4) Global Reactivity Attention: 全局注意力交互
  │
  ├── (5) unbatch_feats: 拆分回原子/键特征
  │
  ├── (6a) atom_linear: 原子分类 → [n_atoms, 125]
  │
  └── (6b) bond_linear: 键分类 → [n_bonds, 549]
  
输出: atom_outs, bond_outs, attention_score
```

### LocalRetro 源码对应
- `scripts/models.py` → `LocalRetro_model`

In [9]:
# ====== 7. 完整 LocalRetro 模型 ======
# 对应 LocalRetro 源码: scripts/models.py → LocalRetro_model

class LocalRetro_model(nn.Module):
    """
    完整的 LocalRetro 模型。
    
    对应 LocalRetro 源码: scripts/models.py → LocalRetro_model
    
    架构:
    1. MPNN: 消息传递图神经网络 → 原子嵌入
    2. linearB: 键特征构建（拼接 + 降维）
    3. GRA: 全局反应性注意力
    4. atom_linear / bond_linear: 分类头
    """
    def __init__(self, node_in_feats, edge_in_feats, node_out_feats,
                 edge_hidden_feats, num_step_message_passing,
                 attention_heads, attention_layers,
                 AtomTemplate_n, BondTemplate_n, activation='gelu'):
        super(LocalRetro_model, self).__init__()
        
        if activation in ['GELU', 'gelu']:
            self.activation = GELU()
        elif activation in ['ReLU', 'relu']:
            self.activation = nn.ReLU()
        
        # (1) MPNN 编码器
        self.mpnn = MPNNGNN(
            node_in_feats=node_in_feats,
            node_out_feats=node_out_feats,
            edge_in_feats=edge_in_feats,
            edge_hidden_feats=edge_hidden_feats,
            num_step_message_passing=num_step_message_passing
        )
        
        # (2) 键特征降维
        self.linearB = nn.Linear(node_out_feats * 2, node_out_feats)
        
        # (3) 全局反应性注意力
        self.att = Global_Reactivity_Attention(
            node_out_feats, attention_heads, attention_layers,
            activation=self.activation
        )
        
        # (4) 原子分类头
        self.atom_linear = nn.Sequential(
            nn.Linear(node_out_feats, node_out_feats),
            self.activation,
            nn.Dropout(0.2),
            nn.Linear(node_out_feats, AtomTemplate_n + 1)
        )
        
        # (5) 键分类头
        self.bond_linear = nn.Sequential(
            nn.Linear(node_out_feats, node_out_feats),
            self.activation,
            nn.Dropout(0.2),
            nn.Linear(node_out_feats, BondTemplate_n + 1)
        )
    
    def forward(self, g, node_feats, edge_feats):
        """
        前向传播。
        
        输入:
            g: DGL batch 图
            node_feats: 节点特征 [total_atoms, node_in_feats]
            edge_feats: 边特征 [total_edges, edge_in_feats]
        
        输出:
            atom_outs: 原子 logits [total_atoms, AtomTemplate_n+1]
            bond_outs: 键 logits [total_bonds, BondTemplate_n+1]
            attention_score: 注意力权重（可用于可视化）
        """
        # (1) MPNN 编码
        node_feats = self.mpnn(g, node_feats, edge_feats)
        atom_feats = node_feats
        
        # (2) 键特征构建
        bond_feats = self.linearB(pair_atom_feats(g, node_feats))
        
        # (3) 组装编辑特征 + 掩码
        edit_feats, mask = unbatch_mask(g, atom_feats, bond_feats)
        
        # (4) 全局反应性注意力
        attention_score, edit_feats = self.att(edit_feats, mask)
        
        # (5) 拆分回原子/键特征
        atom_feats, bond_feats = unbatch_feats(g, edit_feats)
        
        # (6) 分类预测
        atom_outs = self.atom_linear(atom_feats)
        bond_outs = self.bond_linear(bond_feats)
        
        return atom_outs, bond_outs, attention_score

# 创建模型实例
model = LocalRetro_model(
    node_in_feats=NODE_FEAT_DIM,
    edge_in_feats=EDGE_FEAT_DIM,
    node_out_feats=NODE_OUT_FEATS,
    edge_hidden_feats=EDGE_HIDDEN_FEATS,
    num_step_message_passing=NUM_STEP_MESSAGE_PASSING,
    attention_heads=ATTENTION_HEADS,
    attention_layers=ATTENTION_LAYERS,
    AtomTemplate_n=ATOM_TEMPLATE_N,
    BondTemplate_n=BOND_TEMPLATE_N,
    activation='gelu'
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LocalRetro 模型创建成功!')
print(f'  总参数量: {total_params:,}')
print(f'  可训练参数: {trainable_params:,}')

# 完整前向传播测试
with torch.no_grad():
    bg_test = bg.to(DEVICE)
    node_f = bg_test.ndata.pop('h').to(DEVICE)
    edge_f = bg_test.edata.pop('e').to(DEVICE)
    atom_out, bond_out, att_scores = model(bg_test, node_f, edge_f)

print(f'\n完整前向传播:')
print(f'  原子输出: {atom_out.shape}')
print(f'  键输出:   {bond_out.shape}')
print(f'  注意力:   {att_scores[0].shape}')

LocalRetro 模型创建成功!
  总参数量: 8,645,410
  可训练参数: 8,645,410

完整前向传播:
  原子输出: torch.Size([16, 125])
  键输出:   torch.Size([32, 549])
  注意力:   torch.Size([2, 8, 30, 30])


---

## 8. 推理流程端到端演示

LocalRetro 的推理流程与 GLN 有本质不同：

### GLN 推理: 级联 Beam Search
```
产物 → 候选中心 → Top-K 中心 → 候选模板 → Top-K 模板 → 反应物验证
```

### LocalRetro 推理: 直接预测 + 解码
```
产物 → MPNN+GRA → 原子/键 logits → Top-K 编辑 → 模板解码 → 反应物
```

具体步骤：

1. **模型预测**: 对产物的每个原子/键输出模板概率分布
2. **合并排序**: 将原子预测和键预测合并，按概率排序
3. **Top-K 选择**: 选择概率最高的 K 个编辑操作
4. **模板解码**: 
   a. 根据编辑类型和位点，查找对应模板
   b. 将模板 SMARTS 应用到产物上（rdChemReactions）
   c. 修正反应物的 H 数/电荷/手性

### LocalRetro 源码对应
- `scripts/get_edit.py` → `output2edit()`, `combined_edit()`, `write_edits()`
- `scripts/Decode_predictions.py` → `get_k_predictions()`
- `LocalTemplate/template_decoder.py` → `decode_localtemplate()`

In [10]:
# ====== 8.1 编辑预测提取 ======
# 对应 LocalRetro 源码: scripts/get_edit.py

def get_id_template(a, class_n):
    """
    将扁平化的索引 a 解码为 (编辑位点, 模板类别)。
    
    对应 LocalRetro 源码: get_edit.py → get_id_template()
    
    逻辑: 输出矩阵 shape = [n_sites, n_templates]
    扁平化后 a = site_idx * n_templates + template_idx
    """
    edit_idx = a // class_n
    template = a % class_n
    return (edit_idx, template)

def output2edit(out, top_num):
    """
    从模型输出中提取 Top-K 编辑预测。
    
    对应 LocalRetro 源码: get_edit.py → output2edit()
    
    步骤:
    1. 将输出矩阵扁平化
    2. 按概率降序排列
    3. 过滤掉 template=0（无编辑）
    4. 取 Top-K
    """
    class_n = out.size(-1)
    readout = out.cpu().detach().numpy().reshape(-1)
    output_rank = np.flip(np.argsort(readout))
    # 过滤 template=0 (无编辑)
    output_rank = [r for r in output_rank 
                   if get_id_template(r, class_n)[1] != 0][:top_num]
    
    selected_edit = [get_id_template(a, class_n) for a in output_rank]
    selected_proba = [readout[a] for a in output_rank]
    return selected_edit, selected_proba

def combined_edit(graph, atom_out, bond_out, top_num):
    """
    合并原子预测和键预测，统一排序。
    
    对应 LocalRetro 源码: get_edit.py → combined_edit()
    """
    edit_id_a, edit_proba_a = output2edit(atom_out, top_num)
    edit_id_b, edit_proba_b = output2edit(bond_out, top_num)
    
    edit_id_c = edit_id_a + edit_id_b
    edit_type_c = ['a'] * len(edit_id_a) + ['b'] * len(edit_id_b)
    edit_proba_c = edit_proba_a + edit_proba_b
    
    # 按概率排序
    edit_rank_c = np.flip(np.argsort(edit_proba_c))[:top_num]
    
    return (
        [edit_type_c[r] for r in edit_rank_c],
        [edit_id_c[r] for r in edit_rank_c],
        [edit_proba_c[r] for r in edit_rank_c]
    )

# 演示
print('编辑预测提取演示')
print('=' * 60)

# 模拟模型输出 (随机初始化模型的预测)
demo_smiles = 'CC(=O)Nc1ccccc1'
demo_graph = smiles_to_graph(demo_smiles, 
                             node_featurizer=node_featurizer,
                             edge_featurizer=edge_featurizer,
                             canonical_atom_order=False)
demo_bg = dgl.batch([demo_graph]).to(DEVICE)

with torch.no_grad():
    nf = demo_bg.ndata.pop('h').to(DEVICE)
    ef = demo_bg.edata.pop('e').to(DEVICE)
    a_out, b_out, _ = model(demo_bg, nf, ef)
    
    # Softmax 归一化
    a_out = nn.Softmax(dim=1)(a_out)
    b_out = nn.Softmax(dim=1)(b_out)

sg = demo_bg.remove_self_loop()
graphs = dgl.unbatch(sg)
n_atoms = graphs[0].num_nodes()
n_bonds = graphs[0].num_edges()

pred_types, pred_sites, pred_scores = combined_edit(
    graphs[0], a_out[:n_atoms], b_out[:n_bonds], top_num=5)

print(f'产物: {demo_smiles}')
print(f'原子数: {n_atoms}, 键数: {n_bonds}')
print(f'\nTop-5 预测编辑:')
for i, (t, s, score) in enumerate(zip(pred_types, pred_sites, pred_scores)):
    type_name = '原子编辑' if t == 'a' else '键编辑'
    print(f'  #{i+1}: {type_name} | 位点={s[0]}, 模板={s[1]} | 分数={score:.4f}')
print('\n(注意: 这是未训练模型的随机预测，仅展示推理流程)')

编辑预测提取演示
产物: CC(=O)Nc1ccccc1
原子数: 10, 键数: 20

Top-5 预测编辑:
  #1: 原子编辑 | 位点=1, 模板=82 | 分数=0.0167
  #2: 原子编辑 | 位点=0, 模板=82 | 分数=0.0155
  #3: 原子编辑 | 位点=2, 模板=82 | 分数=0.0154
  #4: 原子编辑 | 位点=2, 模板=120 | 分数=0.0153
  #5: 原子编辑 | 位点=3, 模板=68 | 分数=0.0152

(注意: 这是未训练模型的随机预测，仅展示推理流程)


In [11]:
# ====== 8.2 模板解码 ======
# 对应 LocalRetro 源码: LocalTemplate/template_decoder.py

def decode_localtemplate_demo(product_smiles, pred_type, pred_site, pred_template_class,
                              atom_templates, bond_templates, template_infos):
    """
    将预测的编辑操作解码为反应物 SMILES。
    
    对应 LocalRetro 源码: template_decoder.py → decode_localtemplate()
    
    简化版，用于教学演示。
    
    步骤:
    1. 根据 pred_type 和 pred_template_class 查找模板 SMARTS
    2. 用 rdChemReactions 将模板应用到产物
    3. 修正反应物的 H 数/电荷/手性
    4. 返回反应物 SMILES
    """
    mol = Chem.MolFromSmiles(product_smiles)
    if mol is None:
        return None
    
    # 查找模板
    if pred_type == 'a':
        if pred_template_class not in atom_templates:
            return None
        template = atom_templates[pred_template_class]
    else:
        if pred_template_class not in bond_templates:
            return None
        template = bond_templates[pred_template_class]
    
    # 提取 SMARTS（模板格式: SMARTS_Hcode_Ccode）
    template_smarts = template.split('_')[0]
    
    # 构建反应 SMARTS
    parts = template_smarts.split('>>')
    if len(parts) != 2:
        return None
    rxn_smarts = '(%s)>>(%s)' % (parts[0], parts[1])
    
    try:
        rxn = rdChemReactions.ReactionFromSmarts(rxn_smarts)
        # 设置原子映射
        [atom.SetAtomMapNum(atom.GetIdx()) for atom in mol.GetAtoms()]
        
        results = rxn.RunReactants([mol])
        if not results:
            return None
        
        # 取第一个结果
        reactants = results[0]
        smiles_parts = []
        for r in reactants:
            [atom.SetAtomMapNum(0) for atom in r.GetAtoms()]
            smi = Chem.MolToSmiles(r)
            if smi:
                smiles_parts.append(smi)
        
        return '.'.join(sorted(smiles_parts)) if smiles_parts else None
    except Exception as e:
        return None

# ====== 完整推理演示 ======
print('完整推理流程演示')
print('=' * 60)

# 使用预定义的模板做演示
demo_atom_templates = {
    1: '[N:1]>>O=C(-[N:1])-O-C-c1:c:c:c:c:c:1_-1_0',  # Boc 保护
}
demo_bond_templates = {
    1: '[N:1]-[C:2]>>[N:1].[O:99]-[C:2]_10_00',   # N-C 键断裂
    2: '[C:1]-[O:2]>>[C:1].[HO:2]_10_00',           # C-O 键断裂
}

product = 'CC(=O)Nc1ccccc1'

print(f'输入产物: {product}')
print(f'\n可用模板:')
for k, v in demo_atom_templates.items():
    print(f'  原子模板 {k}: {v}')
for k, v in demo_bond_templates.items():
    print(f'  键模板 {k}: {v}')

# 尝试解码
print(f'\n尝试应用各模板:')
for tpl_class, tpl in demo_bond_templates.items():
    result = decode_localtemplate_demo(
        product, 'b', 0, tpl_class, 
        demo_atom_templates, demo_bond_templates, {})
    if result:
        print(f'  键模板 {tpl_class}: {product} → {result}')
    else:
        print(f'  键模板 {tpl_class}: 无法匹配')

完整推理流程演示
输入产物: CC(=O)Nc1ccccc1

可用模板:
  原子模板 1: [N:1]>>O=C(-[N:1])-O-C-c1:c:c:c:c:c:1_-1_0
  键模板 1: [N:1]-[C:2]>>[N:1].[O:99]-[C:2]_10_00
  键模板 2: [C:1]-[O:2]>>[C:1].[HO:2]_10_00

尝试应用各模板:
  键模板 1: CC(=O)Nc1ccccc1 → CC(=O)O.Nc1ccccc1
  键模板 2: 无法匹配


---

## 9. 训练流程概览

### 训练循环

```python
for epoch in range(num_epochs):
    for batch in train_loader:
        smiles, bg, atom_labels, bond_labels = batch
        
        # 前向传播
        atom_logits, bond_logits, _ = model(bg, bg.ndata['h'], bg.edata['e'])
        
        # 计算损失
        loss_atom = CrossEntropyLoss(atom_logits, atom_labels)  # 原子分类损失
        loss_bond = CrossEntropyLoss(bond_logits, bond_labels)  # 键分类损失
        total_loss = concat(loss_atom, loss_bond).mean()
        
        # 反向传播
        total_loss.backward()
        clip_grad_norm_(model.parameters(), max_clip=20)
        optimizer.step()
```

### 训练配置 (default_config.json)

| 参数 | 值 | 说明 |
|------|-----|------|
| node_out_feats | 320 | MPNN 输出维度 |
| edge_hidden_feats | 64 | 边 MLP 隐藏维度 |
| num_step_message_passing | 6 | 消息传递步数 |
| attention_heads | 8 | 注意力头数 |
| attention_layers | 1 | 注意力层数 |
| batch_size | 16 | 批大小 |
| learning_rate | 1e-4 | 学习率 |
| num_epochs | 50 | 最大训练轮数 |
| patience | 5 | 早停耐心值 |
| max_clip | 20 | 梯度裁剪 |

### LocalRetro 源码对应
- `scripts/Train.py` → `run_a_train_epoch()`, `run_an_eval_epoch()`, `main()`

In [12]:
# ====== 9. 训练流程演示 (微型数据) ======
# 对应 LocalRetro 源码: scripts/Train.py

from torch.optim import Adam

# 构造教学用的微型 batch
demo_smiles_list = ['CC(=O)Nc1ccccc1', 'c1ccc(-c2ccccc2)cc1']
demo_graphs = [smiles_to_graph(s, node_featurizer=node_featurizer, 
                                edge_featurizer=edge_featurizer,
                                canonical_atom_order=False) for s in demo_smiles_list]
demo_bg = dgl.batch(demo_graphs).to(DEVICE)

# 构造标签（大部分为0=无编辑）
n_total_atoms = demo_bg.num_nodes()
n_total_bonds = demo_bg.remove_self_loop().num_edges()
demo_atom_labels = torch.zeros(n_total_atoms, dtype=torch.long).to(DEVICE)
demo_bond_labels = torch.zeros(n_total_bonds, dtype=torch.long).to(DEVICE)

# 给第一个分子的第3个原子标注 template 1 (模拟 N-H 编辑)
demo_atom_labels[3] = 1

# 创建训练组件
model_train = LocalRetro_model(
    node_in_feats=NODE_FEAT_DIM,
    edge_in_feats=EDGE_FEAT_DIM,
    node_out_feats=NODE_OUT_FEATS,
    edge_hidden_feats=EDGE_HIDDEN_FEATS,
    num_step_message_passing=NUM_STEP_MESSAGE_PASSING,
    attention_heads=ATTENTION_HEADS,
    attention_layers=ATTENTION_LAYERS,
    AtomTemplate_n=ATOM_TEMPLATE_N,
    BondTemplate_n=BOND_TEMPLATE_N,
    activation='gelu'
).to(DEVICE)

optimizer = Adam(model_train.parameters(), lr=1e-4, weight_decay=1e-6)
loss_criterion = nn.CrossEntropyLoss(reduction='none')

print('训练演示 (10 个 epoch，2 个分子/batch)')
print('=' * 50)

losses = []
for epoch in range(10):
    model_train.train()
    
    node_feats = demo_bg.ndata['h'].to(DEVICE)
    edge_feats = demo_bg.edata['e'].to(DEVICE)
    
    # 注意: DGL 的 pop 会从图中移除特征，所以这里用索引方式
    atom_logits, bond_logits, _ = model_train(demo_bg, node_feats, edge_feats)
    
    # 计算损失
    loss_a = loss_criterion(atom_logits, demo_atom_labels)
    loss_b = loss_criterion(bond_logits, demo_bond_labels)
    total_loss = torch.cat([loss_a, loss_b]).mean()
    
    # 反向传播
    optimizer.zero_grad()
    total_loss.backward()
    nn.utils.clip_grad_norm_(model_train.parameters(), max_norm=20)
    optimizer.step()
    
    losses.append(total_loss.item())
    
    # 重新设置图特征（因为前向传播中 pop 了特征）
    demo_bg = dgl.batch(demo_graphs).to(DEVICE)
    
    if (epoch + 1) % 2 == 0:
        print(f'  Epoch {epoch+1:3d} | Loss: {total_loss.item():.4f}')

print(f'\n训练完成! 初始损失: {losses[0]:.4f} → 最终损失: {losses[-1]:.4f}')

训练演示 (10 个 epoch，2 个分子/batch)
  Epoch   2 | Loss: 5.1169
  Epoch   4 | Loss: 4.2171
  Epoch   6 | Loss: 3.5025
  Epoch   8 | Loss: 3.0019
  Epoch  10 | Loss: 2.4598

训练完成! 初始损失: 5.8079 → 最终损失: 2.4598


---

## 10. 总结

### 模型架构回顾

```
LocalRetro 完整计算流程
══════════════════════════════════════════════════════════════

输入: 产物 SMILES
  │
  ├── DGLLife 特征化
  │   ├── WeaveAtomFeaturizer → 节点特征 [n, 27]
  │   └── CanonicalBondFeaturizer → 边特征 [m, 12]
  │
  ├── MPNN (6步消息传递)
  │   └── 原子嵌入 [n, 320]
  │
  ├── pair_atom_feats + linearB
  │   └── 键嵌入 [m', 320]  (m' = m - n, 去自环)
  │
  ├── unbatch_mask
  │   └── 编辑特征 [batch, max_seq, 320] + mask
  │
  ├── Global Reactivity Attention (8头 × 1层)
  │   └── 增强编辑特征 [batch, max_seq, 320]
  │
  ├── unbatch_feats
  │   ├── 原子特征 [n, 320]
  │   └── 键特征 [m', 320]
  │
  ├── atom_linear → [n, 125]  (124 原子模板 + 无编辑)
  └── bond_linear → [m', 549] (548 键模板 + 无编辑)
```

### 与 GLN 的完整对比

| 方面 | GLN | LocalRetro |
|------|-----|------------|
| 图框架 | torch-geometric | **DGL** |
| GNN | Mean-Field GNN | **MPNN** (Edge-based) |
| 注意力 | 产物-候选项 匹配注意力 | **全局自注意力** (Transformer) |
| 模板粒度 | 全局（整个反应） | **局部**（原子/键） |
| 预测方式 | 层次化 3 级概率 | **直接多分类** |
| 推理 | Beam Search | **Top-K 编辑** |
| 解码 | rdChiral 正向应用 | **rdChemReactions + 属性修正** |
| 优势 | 结构化概率推理 | **简单高效, 可解释编辑** |

### 各组件参数量 (USPTO_50K)

| 组件 | 参数量 |
|------|--------|
| MPNN | ~1.3M |
| linearB | ~205K |
| GRA (Attention) | ~1.2M |
| atom_linear | ~143K |
| bond_linear | ~282K |
| **总计** | **~3.1M** |

### 性能参考 (USPTO_50K, Top-1 Accuracy)

| 方法 | Top-1 |
|------|-------|
| GLN | 52.5% |
| **LocalRetro** | **53.4%** |

> LocalRetro 以更简洁的架构取得了更好的性能，其核心优势在于：局部模板更精细、Global Reactivity Attention 捕捉远程效应、直接分类避免了级联误差。